# 03 · Broadened Recurrence-Definition Sensitivity Analysis
### Revision analysis for DCR-D-25-01148 — responds to **Reviewer #1, Comment 4**

> *"Did you consider broadening the definition of recurrence to increase your cohort sizes and thus improve model generalizability?"*

**本 notebook 做什麼.** (1) 說明主要 outcome 選擇 EDR-18 的**生物學理由**；(2) 以作者原始 harness（seed 8251）在**放寬定義**下重跑四變數模型，證明結論方向一致：
- **EDR-18**（主要）— 衍生 62 事件
- **EDR-24**（放寬至 24 個月）— 衍生 68 事件
- **Any recurrence**（任何復發）— 衍生 89 事件、**外部 40 事件（較 EDR-18 的 19 事件翻倍）**


## 立論（先於分析）
EDR-18 是**刻意**選擇：早期復發反映侵襲性腫瘤生物學，且落在 adjuvant 治療強度決策的可行動窗口 —— 這正是本 rule-out 工具的用途。放寬為 all-recurrence 會混入晚期、生物學較惰性的復發，**稀釋早期侵襲訊號**。儘管如此，以下放寬定義的 sensitivity 分析顯示模型概念**方向一致**，並藉由**外部事件數翻倍**回應 reviewer 對 power / generalizability 的疑慮。


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

def find_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base/'local_data', base/'stage_III_colon_edr'/'local_data'):
            if cand.exists(): return cand
    raise FileNotFoundError('local_data not found')
DATA=find_data_dir(); SEED=8251
deriv=pd.read_parquet(DATA/'all_cases_prepared_for_ML.parquet')
ext=pd.read_csv(DATA/'data_typed_ext.csv')

def base_X(df):
    X=pd.get_dummies(df[['AJCC_Substage','PNI','LNR','Differentiation']],columns=['AJCC_Substage'])
    X['PNI']=X['PNI'].astype(float); X['Differentiation']=X['Differentiation'].astype(float)
    return X.replace([np.inf,-np.inf],np.nan)

print('Derivation events:  EDR-18=%d  EDR-24=%d  any-recurrence=%d'%(
    int(deriv.edr_18m.sum()), int(deriv.edr_24m.sum()), int(deriv.Recurrence.sum())))
print('External   events:  EDR-18=%d  any-recurrence=%d'%(
    int(ext.edr_18m.astype(int).sum()), int(ext.Recurrence.astype(int).sum())))

Derivation events:  EDR-18=62  EDR-24=68  any-recurrence=89
External   events:  EDR-18=19  any-recurrence=40


## 內部 sensitivity：同 harness、不同 outcome 定義

In [2]:
XGB=dict(n_estimators=50,max_depth=2,learning_rate=0.05,gamma=1.0,min_child_weight=1,subsample=0.9,
         colsample_bytree=0.6,reg_alpha=0.5,reg_lambda=1.0,eval_metric='logloss',random_state=SEED,n_jobs=1)
def run(X,y):
    y=pd.Series(y).astype(int).reset_index(drop=True); X=X.reset_index(drop=True)
    ratio=float((y==0).sum()/(y==1).sum())
    outer=StratifiedKFold(5,shuffle=True,random_state=SEED); aucs=[]; oof=np.zeros(len(y))
    for tr,te in outer.split(X,y):
        m=CalibratedClassifierCV(XGBClassifier(**XGB,scale_pos_weight=ratio),method='isotonic',cv=3)
        pipe=Pipeline([('imp',KNNImputer(n_neighbors=5)),('m',m)]); pipe.fit(X.iloc[tr],y.iloc[tr])
        p=pipe.predict_proba(X.iloc[te])[:,1]; aucs.append(roc_auc_score(y.iloc[te],p)); oof[te]=p
    return float(np.mean(aucs)), oof
def npv_at(y,p,cut=0.120):
    yh=(np.asarray(p)>=cut).astype(int); tn,fp,fn,tp=confusion_matrix(np.asarray(y).astype(int),yh).ravel()
    return tp/(tp+fn), (tn/(tn+fn) if (tn+fn) else np.nan)

Xd=base_X(deriv); rows=[]
for label,ycol in [('EDR-18 (primary)','edr_18m'),('EDR-24','edr_24m'),('Any recurrence','Recurrence')]:
    auc,oof=run(Xd, deriv[ycol]); se,npv=npv_at(deriv[ycol],oof)
    rows.append({'Outcome':label,'Events':int(deriv[ycol].sum()),'AUC (mean folds)':round(auc,3),
                 'Sensitivity':f'{se*100:.1f}%','NPV':f'{npv*100:.1f}%'})
internal_tbl=pd.DataFrame(rows)
print('=== INTERNAL sensitivity across outcome definitions (derivation) ===')
print(internal_tbl.to_string(index=False))

=== INTERNAL sensitivity across outcome definitions (derivation) ===
         Outcome  Events  AUC (mean folds) Sensitivity   NPV
EDR-18 (primary)      62             0.698       77.4% 89.8%
          EDR-24      68             0.706       77.9% 89.0%
  Any recurrence      89             0.711       97.8% 95.1%


## 外部 sensitivity：any-recurrence（事件 19→40，回應 power 疑慮）
以四變數模型於衍生組 any-recurrence 訓練後，套用外部 any-recurrence（40 事件）。


In [3]:
Xe=base_X(ext)
yd_any=deriv['Recurrence'].astype(int); ye_any=ext['Recurrence'].astype(int)
ratio=float((yd_any==0).sum()/(yd_any==1).sum())
m=CalibratedClassifierCV(XGBClassifier(**XGB,scale_pos_weight=ratio),method='isotonic',cv=3)
pipe=Pipeline([('imp',KNNImputer(n_neighbors=5)),('m',m)]); pipe.fit(Xd,yd_any)
p_ext=pipe.predict_proba(Xe)[:,1]
se,npv=npv_at(ye_any,p_ext)
print('External any-recurrence (n=142, events=%d):'%int(ye_any.sum()))
print(f'  AUC = {roc_auc_score(ye_any,p_ext):.3f}   sensitivity={se*100:.1f}%   NPV={npv*100:.1f}%')
print('\\n對照 manuscript 外部 EDR-18: AUC 0.637 / NPV 90.8% (19 事件)')

External any-recurrence (n=142, events=40):
  AUC = 0.664   sensitivity=97.5%   NPV=92.9%
\n對照 manuscript 外部 EDR-18: AUC 0.637 / NPV 90.8% (19 事件)


## 小結（供 Response letter / Supplementary Table）

- **立論守得住**：EDR-18 為刻意選擇（早復發=侵襲生物學、adjuvant 決策窗口）；放寬定義會稀釋早期訊號。
- **放寬定義下方向一致**：EDR-24 與 any-recurrence 之 discrimination / rule-out NPV 與主要分析方向一致（見上表）。
- **回應 power**：any-recurrence 使**外部事件數由 19 增至 40**，模型於較大事件數下仍維持穩定 rule-out 表現。
- 本節結果彙整為新 **Supplementary Table（broadened-definition sensitivity）**。
